In [1]:
import timetree_exporter

In [2]:
from datetime import date, datetime, time, timedelta
from pathlib import Path
from zoneinfo import ZoneInfo
from icalendar import Calendar

INPUT = Path("ics_files/output.ics")
OUTPUT = Path("ics_files/filtered.ics")
TARGET_DAY = date(2026, 8, 3)  # <-- change this


def event_occurs_on(component, day: date) -> bool:
    """True if the VEVENT overlaps `day` (local calendar date)."""
    start = component.decoded("dtstart")
    end = component.decoded("dtend")

    # All-day: DATE values; DTEND is exclusive in ICS
    if isinstance(start, date) and not isinstance(start, datetime):
        return start <= day < end

    # Timed: use the event's own timezone (or UTC if none)
    tz = start.tzinfo or ZoneInfo("UTC")
    day_start = datetime.combine(day, time.min, tzinfo=tz)
    day_end = day_start + timedelta(days=1)
    return start < day_end and end > day_start


cal = Calendar.from_ical(INPUT.read_bytes())

filtered = Calendar()
# for name, value in cal.property_items():
#     if name not in ("BEGIN", "END"):
#         filtered.add(name, value)

# Keep timezone defs used by matched events
kept = []
used_tzids = set()
for component in cal.walk():
    if component.name != "VEVENT":
        continue
    if event_occurs_on(component, TARGET_DAY):
        kept.append(component)
        dtstart = component.get("dtstart")
        if dtstart is not None and "TZID" in dtstart.params:
            used_tzids.add(dtstart.params["TZID"])

for component in cal.walk():
    if component.name == "VTIMEZONE" and str(component.get("tzid")) in used_tzids:
        filtered.add_component(component)

for event in kept:
    filtered.add_component(event)

OUTPUT.write_bytes(filtered.to_ical())
print(f"{len(kept)} events on {TARGET_DAY} → {OUTPUT}")
for e in kept:
    print("-", e.get("summary"), "|", e.get("dtstart"), "→", e.get("dtend"))

3 events on 2026-08-03 → ics_files/filtered.ics
- [OIL] Xavier Kang | vDDDTypes(2026-08-03, Parameters({})) → vDDDTypes(2026-08-04, Parameters({}))
- ATP (SP Coy) | vDDDTypes(2026-08-03, Parameters({})) → vDDDTypes(2026-08-04, Parameters({}))
- Off for P2 | vDDDTypes(2026-08-03, Parameters({})) → vDDDTypes(2026-08-04, Parameters({}))


In [3]:
import gspread
import pandas as pd
from google.oauth2.service_account import Credentials

SHEET_NAME = "0. 10th Gen 1st Coy Tracking caa 280726"
headers = ["S/N", "PLT", "NRIC", "RANK", "NAME", "CONTACT NUMBER", "DOB", "DOE", "ORD", "PES", "BLOOD TYPE", "EMERGENCY CONTACT", "RELATIONSHIP", "E CONTACT NO", "CPR AED EXPIRY", "CPR REFRESHER DATE"]

scopes = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]
creds = Credentials.from_service_account_file("credentials.json", scopes=scopes)
client = gspread.authorize(creds)

sheet = client.open(SHEET_NAME).sheet1
df = pd.DataFrame(sheet.get_all_records(head=2))
print(df.head())

     S/N  PLT       NRIC RANK                NAME CONTACT NO        DOB  \
0      1   HQ  S9711986D  CPT  PAMELA KOH RONG YI  9839 0924  15 Apr 97   
1      2   HQ  S9404778A  3WO  SIM ZHAO MING SEAN  8742 1716  03 Feb 94   
2      3    1  T0119270H  LTA  JOLENE TEO MIAN EN  8444 4722  29 Jun 01   
3      4    1  T0609684G  2LT        NEO HUI RONG  8784 8748  15 Apr 06   
4      5    1  T0503011G  2LT        CHEW JIA JIE   96491866   7 Feb 05   

         DOE        ORD PES BLOOD TYPE   EMERGENCY CONTACT RELATIONSHIP  \
0  05 Jan 17  15 Apr 47   A         O+        GOH JUN WANG       SPOUSE   
1  27 May 15   3 Feb 49  B1         A+  SIM ZHAO WEN BRYAN      BROTHER   
2   4 Jan 20  29 Jun 51   A         A+   BRYAN HAN ZI YING       SPOUSE   
3   6 Jan 25   5 Nov 26   A         A+         NEO SAW YAW       FATHER   
4  02 Jul 25   1 May 27  B1         A+      KOH SHIANG WEE       MOTHER   

  E CONTACT NO CPR AED EXPIRY CPR REFRESHER DATE  
0    9223 5920       8 Jan 27                  

In [24]:
df.columns = df.columns.str.strip()

In [27]:
rel_cols = ['PLT', 'NRIC', 'RANK', 'NAME', 'CONTACT NO']
sdf = df[rel_cols]
sdf.head(20)

,PLT,NRIC,RANK,NAME,CONTACT NO
0,HQ,S9711986D,CPT,PAMELA KOH RONG YI,9839 0924
1,HQ,S9404778A,3WO,SIM ZHAO MING SEAN,8742 1716
2,1,T0119270H,LTA,JOLENE TEO MIAN EN,8444 4722
3,1,T0609684G,2LT,NEO HUI RONG,8784 8748
4,1,T0503011G,2LT,CHEW JIA JIE,96491866
5,1,T0114013I,2SG,"CHUA CHIN WEI,CLEDWYN",8692 6435
6,1,T0672025G,2SG,CHENG KAI YANG,9028 8122
7,1,T0525842H,3SG,LINUS CHIA MING RONG,8784 9001
8,1,T0609967F,3SG,TIMOTHY TENG YU FENG,8613 6685
9,2,T0432945C,LTA,LEE ZI FENG MATIAS,8869 2411


## Experimenting with the syntax and functions

In [4]:
client.open(SHEET_NAME).worksheets()

[<Worksheet 'Comd' id:774101508>,
 <Worksheet 'Soldiers' id:1940727009>,
 <Worksheet 'BUC' id:0>,
 <Worksheet 'HR' id:730128688>,
 <Worksheet 'O/L Summary' id:7778783>,
 <Worksheet 'Earned O/L' id:681662043>,
 <Worksheet 'Duty Points' id:1843729956>,
 <Worksheet 'Duty List' id:955234911>,
 <Worksheet 'Fines' id:173077696>,
 <Worksheet 'GD Points' id:1398276808>,
 <Worksheet 'Taskings' id:1315778445>,
 <Worksheet 'ROA' id:560888919>,
 <Worksheet 'Sizing' id:172953078>,
 <Worksheet 'Dropdown' id:112066808>,
 <Worksheet 'MSP' id:2096185282>,
 <Worksheet 'Static Tracking' id:1149731376>]

In [5]:
sh = client.open(SHEET_NAME)
hr_sheet = sh.worksheet('HR')

In [10]:
trunc_hr_sheet = hr_sheet.get("B4:N61")
hr_df = pd.DataFrame(trunc_hr_sheet)

In [11]:
hr_df

,0,1,2,3,4,5,6,7,8,9,10,11,12
0,RANK,NAME,Status 1,Status 2,Status 3,Status 4,Status 4,Status 5,Status 6,Status 7,Status 8,Status 9,Status 10
1,PTE,TAN AH GAO,Reason / Symptoms\n3D MC (DDMMYY - DDMMYY),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,PTE,LIM LI DIAN,Fever\n2D LD (160626 - 170626),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,PTE,HE MINGZHE,Fever\n 2D MC (280426 - 290426),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,PTE,MARCUS CHAN RAY MENG,Ear Infection \n4D Ex Loud Noise \n(140526 - 1...,"3D RMJ, EX HL \n(200526 - 220526)",3D Ex RMJ\n(280526 - 300526),1D LD\n(080626),7D Ex Loud Noise\n(080626 - 140626),14D Ex Er Plugs\n(110626 - 240626),14D Ex Loud Environment\n(110626 - 240626),30D Ex Heavy Load\n(090626 - 080726),30D Ex Prolonged Sitting\n(090626 - 080726),30D Ex Prolonged Standing\n(090626 - 080726),7D LD (150726 - 210726)
5,PTE,LIM ZHAN PENG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,PTE,WONG XI JIE,Fever\n2D LD (110626 - 120626),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,PTE,"BOO YI ZHENG, ETHAN",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,PTE,ISAAC CHEONG YI JIE,Corneal Abrasion\n2D MC (260526 - 260626),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,PTE,SOH JIEFENG,Bruised finger\n2D Ex UL (230426 - 240426),Tonsillitis\n1D LD (280726),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
hr_df.ffill(axis=1).iloc[:,-1]

0                                             Status 10
1            Reason / Symptoms\n3D MC (DDMMYY - DDMMYY)
2                        Fever\n2D LD (160626 - 170626)
3                       Fever\n 2D MC (280426 - 290426)
4                               7D LD (150726 - 210726)
5                                         LIM ZHAN PENG
6                        Fever\n2D LD (110626 - 120626)
7                                   BOO YI ZHENG, ETHAN
8             Corneal Abrasion\n2D MC (260526 - 260626)
9                           Tonsillitis\n1D LD (280726)
10                              2D MC (190326 - 200326)
11                                             ZAVE WEE
12                                Fever\n1D LD (200426)
13                                        HONG CHIA KUO
14                                        FOONG QI YANG
15                  Ankle 3D Ex RMJ \n(140726 - 160726)
16           30D Ex Dust Environment\n(280526 - 260626)
17                                   YAN WAI MAN

In [25]:
def extract_rel_entries(sheet):
    RN_COLS = "B4:C61"
    RSI_COLS = "D4:N61"
    RSO_COLS = "P4:T61"
    ranges_data = sheet.batch_get([RN_COLS, RSO_COLS, RSI_COLS])
    rso_df = pd.concat([pd.DataFrame(ranges_data[0]), pd.DataFrame(ranges_data[1])], axis=1)
    rsi_df = pd.concat([pd.DataFrame(ranges_data[0]), pd.DataFrame(ranges_data[2])], axis=1)
    return rso_df, rsi_df


In [3]:
def clean_up(df):
    """Makes first row cols the col name, removes col row and first example, separate"""
    df.columns = df.iloc[0]
    clean_df = df.drop([0, 1])
    return clean_df

In [4]:
def most_recent_status(df):
    names = df[['RANK','NAME']]
    statuses = df[[col for col in list(df.columns) if col not in ['RANK', 'NAME']]]
    temp = statuses.ffill(axis=1).iloc[:,-1]
    new_df = pd.concat([names, temp], axis=1)
    return new_df



In [30]:
clean_df = clean_up(hr_df)
most_recent_status(clean_df)

,RANK,NAME,Status 10
2,PTE,LIM LI DIAN,Fever\n2D LD (160626 - 170626)
3,PTE,HE MINGZHE,Fever\n 2D MC (280426 - 290426)
4,PTE,MARCUS CHAN RAY MENG,7D LD (150726 - 210726)
5,PTE,LIM ZHAN PENG,NaN
6,PTE,WONG XI JIE,Fever\n2D LD (110626 - 120626)
7,PTE,"BOO YI ZHENG, ETHAN",NaN
8,PTE,ISAAC CHEONG YI JIE,Corneal Abrasion\n2D MC (260526 - 260626)
9,PTE,SOH JIEFENG,Tonsillitis\n1D LD (280726)
10,PTE,PERRIN POH YI KAI,2D MC (190326 - 200326)
11,PTE,ZAVE WEE,NaN


In [ ]:
temp = list(client.open(SHEET_NAME).worksheets())
temp[0].title

AttributeError: 'Worksheet' object has no attribute 'name'

In [33]:
import gspread
import pandas as pd
from google.oauth2.service_account import Credentials

def load_sheet(ws_name="HR"):
    SHEET_NAME = "0. 10th Gen 1st Coy Tracking caa 280726"

    scopes = [
        "https://www.googleapis.com/auth/spreadsheets",
        "https://www.googleapis.com/auth/drive",
    ]
    creds = Credentials.from_service_account_file("credentials.json", scopes=scopes)
    client = gspread.authorize(creds)
    sh = client.open(SHEET_NAME)
    temp = [ws.title for ws in list(sh.worksheets())]
    if ws_name not in temp:
        return False, f"NO WS FOUND. WS TITLES ARE {temp}"
    return True, sh.worksheet(ws_name)

def main():
    success, temp = load_sheet()
    if not success:
        return temp
    rso_df, rsi_df = extract_rel_entries(temp)
    clean_rso_df = clean_up(rso_df)
    clean_rsi_df = clean_up(rsi_df)
    rso_active_status, rsi_active_status = get_active_status(clean_rso_df), get_active_status(clean_rsi_df)
    return rso_active_status, rsi_active_status


In [34]:
rso_active, rsi_active = main()

In [35]:
rso_active

,_id,RANK,NAME,active_statuses
0,3,PTE,HE MINGZHE,[RSO Sore Throat \n2D MC (030826-040826)]
1,4,PTE,MARCUS CHAN RAY MENG,[RSO Fever\n3D MC (010826 - 030826)]
2,25,PTE,MURALITHAREN S/O THIAGARAJAN,[RSO Food Poisoning \n3D MC (010826 - 030826)]


In [36]:
rsi_active

,_id,RANK,NAME,active_statuses
0,15,PTE,LUKE JOSHUA LOW,"[84D Ex Firearms\n(050626 - 270826), 84D Ex He..."
1,29,PTE,"LIM EE KEAT, MARC",[30D Dusty Environment \n& Uniform \n(310726-2...


In [27]:
clean_rso_status, clean_rsi_status = main()

In [28]:
import re
from datetime import datetime, date
import pandas as pd

STATUS_RE = re.compile(
    r"(?P<start>\d{6})\s*-\s*(?P<end>\d{6})"
)

def parse_ddmmyy(s: str) -> date:
    return datetime.strptime(s, "%d%m%y").date()

def is_active(cell, on: date) -> bool:
    if pd.isna(cell) or not str(cell).strip():
        return False
    m = STATUS_RE.search(str(cell))
    if not m:
        return False
    start = parse_ddmmyy(m.group("start"))
    end = parse_ddmmyy(m.group("end"))
    return start <= on <= end

def get_active_status(df):
    status_cols = [col for col in df.columns if col not in ['RANK', 'NAME']]
    today = date(2026, 8, 3)  # 030826

    active = (
        df.assign(_id=df.index)
        .melt(id_vars=["_id", "RANK", "NAME"], value_vars=status_cols, var_name="col", value_name="status")
        .loc[lambda d: d["status"].map(lambda x: is_active(x, today))]
    )

    # one row per person with all active statuses that day
    active_by_person = (
        active.groupby(["_id", "RANK", "NAME"])["status"]
        .apply(list)
        .reset_index(name="active_statuses")
    )
    return active_by_person

In [29]:
clean_rso_status.head()

,RANK,NAME,MC 1,MC 2,MC 3,MC 4,MC 5
2,PTE,LIM LI DIAN,NaN,NaN,NaN,NaN,NaN
3,PTE,HE MINGZHE,Sprain Ankle\n7D LD (140526 - 200526),RSO Sore Throat \n2D MC (030826-040826),NaN,NaN,NaN
4,PTE,MARCUS CHAN RAY MENG,RSO Fever\n6D MC (030526 - 080526),RSO Conjunctivitis \n2D MC (020726 - 030726),RSO Insomnia \n2D MC (130726 - 140726),RSO Ear\n1D MC (170726),RSO Fever\n3D MC (010826 - 030826)
5,PTE,LIM ZHAN PENG,RSO Nose Inflammation \n2D MC (060726 - 070726),NaN,NaN,NaN,NaN
6,PTE,WONG XI JIE,NaN,NaN,NaN,NaN,NaN


In [30]:
get_active_status(clean_rso_status)

,_id,RANK,NAME,active_statuses
0,3,PTE,HE MINGZHE,[RSO Sore Throat \n2D MC (030826-040826)]
1,4,PTE,MARCUS CHAN RAY MENG,[RSO Fever\n3D MC (010826 - 030826)]
2,25,PTE,MURALITHAREN S/O THIAGARAJAN,[RSO Food Poisoning \n3D MC (010826 - 030826)]


In [31]:
get_active_status(clean_rsi_status)

,_id,RANK,NAME,active_statuses
0,15,PTE,LUKE JOSHUA LOW,"[84D Ex Firearms\n(050626 - 270826), 84D Ex He..."
1,29,PTE,"LIM EE KEAT, MARC",[30D Dusty Environment \n& Uniform \n(310726-2...


### LLM calling

In [37]:
rso_active_text = "[{'RANK': 'PTE', 'NAME': 'HE MINGZHE', 'active_statuses': ['RSO Sore Throat \n2D MC (030826-040826)']}, {'RANK': 'PTE', 'NAME': 'MARCUS CHAN RAY MENG', 'active_statuses': ['RSO Fever\n3D MC (010826 - 030826)']}, {'RANK': 'PTE', 'NAME': 'MURALITHAREN S/O THIAGARAJAN', 'active_statuses': ['RSO Food Poisoning \n3D MC (010826 - 030826)']}]"
rsi_active_text = "[{'RANK': 'PTE', 'NAME': 'LUKE JOSHUA LOW', 'active_statuses': ['84D Ex Firearms\n(050626 - 270826)', '84D Ex Heavy Machinery \n(050626 - 270826)']}, {'RANK': 'PTE', 'NAME': 'LIM EE KEAT, MARC', 'active_statuses': ['30D Dusty Environment \n& Uniform \n(310726-290826)']}]]"
timetree_text = "[(PosixPath('ics_files/output_ltw.ics'), [{'summary': '[OIL] Xavier Kang', 'start': 'vDDDTypes(2026-08-03, Parameters({}))', 'end': 'vDDDTypes(2026-08-04, Parameters({}))'}, {'summary': 'ATP (SP Coy)', 'start': 'vDDDTypes(2026-08-03, Parameters({}))', 'end': 'vDDDTypes(2026-08-04, Parameters({}))'}, {'summary': 'Off for P2', 'start': 'vDDDTypes(2026-08-03, Parameters({}))', 'end': 'vDDDTypes(2026-08-04, Parameters({}))'}]), (PosixPath('ics_files/output_men.ics'), [])]"

In [38]:
prompt = f"""You are a helpful assistant who keeps track of activities going on in 1st Company, 48 Singapore Armoured Regiment (48SAR). You keep track of statuses (like Medical Certificates ie MCs, excuse Light Duties, ie LDs etc), events going on (like Platoon Cohesion etc), and offs/leaves.

Use the data below to create a summary of things going on today.

Status data: {rso_active_text}\n{rsi_active_text}
Other data: {timetree_text}

Your response should first list out the activities going on today. Then, list out the active statuses (no need to make distinction between RSO and RSI). Include information on the type of status, a description, and the relevant timeframe. Then, list out the personnel on off and leave (include information about when the offs/leaves are until).
"""

In [46]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

# llm = ChatOpenAI(model="openai/gpt-oss-20b", temperature=0.0, openai_api_base="https://router.huggingface.co/v1/", openai_api_key=os.environ.get("HUGGING_FACE_HUB_TOKEN"))

True

In [45]:
llm.invoke(prompt)

2026-08-03 15:13:04 [INFO] HTTP Request: POST https://router.huggingface.co/v1/chat/completions "HTTP/1.1 200 OK" 


AIMessage(content='**Activities scheduled for 3\u202fAug\u202f2026**\n\n| Event | Start | End | Notes |\n|-------|-------|-----|-------|\n| **[OIL] Xavier Kang** | 3\u202fAug\u202f2026 | 4\u202fAug\u202f2026 | Oil‑related duty (OIL) |\n| **ATP (SP Coy)** | 3\u202fAug\u202f2026 | 4\u202fAug\u202f2026 | All‑Troop Parade for the Special Company |\n| **Off for P2** | 3\u202fAug\u202f2026 | 4\u202fAug\u202f2026 | Leave/absence for P2 (no further details available) |\n\n---\n\n### Active statuses (RSO/RSI)\n\n| Rank | Name | Status Type | Description | Timeframe |\n|------|------|-------------|-------------|-----------|\n| PTE | HE\u202fMINGZHE | MC | Sore Throat | 3\u202fAug\u202f2026 – 4\u202fAug\u202f2026 |\n| PTE | MARCUS\u202fCHAN\u202fRAY\u202fMENG | MC | Fever | 1\u202fAug\u202f2026 – 3\u202fAug\u202f2026 |\n| PTE | MURALITHAREN\u202fS/O\u202fTHIAGARAJAN | MC | Food Poisoning | 1\u202fAug\u202f2026 – 3\u202fAug\u202f2026 |\n| PTE | LUKE\u202fJOSHUA\u202fLOW | Ex | Firearms | 5\u202fJu

In [47]:
from openai import AsyncOpenAI

async_client = AsyncOpenAI()
response = await async_client.responses.create(model="gpt-4o-mini", input=prompt)
response

2026-08-03 15:18:44 [INFO] HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK" 


Response(id='resp_04dda6203fba58a0006a7040c3e068819a854399091a1b3f16', created_at=1785741509.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-mini-2024-07-18', object='response', output=[ResponseOutputMessage(id='msg_04dda6203fba58a0006a7040c7678c819aad6275966befc4e3', content=[ResponseOutputText(annotations=[], text='### Activities Going On Today\n\n1. **[OIL] Xavier Kang** \n   - Duration: 2026-08-03 to 2026-08-04\n\n2. **ATP (SP Coy)**\n   - Duration: 2026-08-03 to 2026-08-04\n\n3. **Off for P2**\n   - Duration: 2026-08-03 to 2026-08-04\n\n---\n\n### Active Statuses\n\n1. **PTE HE MINGZHE**\n   - **Type:** Medical Certificate (MC)\n   - **Description:** Sore Throat\n   - **Timeframe:** 030826 to 040826\n\n2. **PTE MARCUS CHAN RAY MENG**\n   - **Type:** Medical Certificate (MC)\n   - **Description:** Fever\n   - **Timeframe:** 010826 to 030826\n\n3. **PTE MURALITHAREN S/O THIAGARAJAN**\n   - **Type:** Medical Certificate (MC)\n   - **Description:*

In [48]:
response.output_text

'### Activities Going On Today\n\n1. **[OIL] Xavier Kang** \n   - Duration: 2026-08-03 to 2026-08-04\n\n2. **ATP (SP Coy)**\n   - Duration: 2026-08-03 to 2026-08-04\n\n3. **Off for P2**\n   - Duration: 2026-08-03 to 2026-08-04\n\n---\n\n### Active Statuses\n\n1. **PTE HE MINGZHE**\n   - **Type:** Medical Certificate (MC)\n   - **Description:** Sore Throat\n   - **Timeframe:** 030826 to 040826\n\n2. **PTE MARCUS CHAN RAY MENG**\n   - **Type:** Medical Certificate (MC)\n   - **Description:** Fever\n   - **Timeframe:** 010826 to 030826\n\n3. **PTE MURALITHAREN S/O THIAGARAJAN**\n   - **Type:** Medical Certificate (MC)\n   - **Description:** Food Poisoning\n   - **Timeframe:** 010826 to 030826\n\n4. **PTE LUKE JOSHUA LOW**\n   - **Type:** Exercise Participation\n   - **Description:** Ex Firearms\n   - **Timeframe:** 050626 to 270826\n\n5. **PTE LUKE JOSHUA LOW**\n   - **Type:** Exercise Participation\n   - **Description:** Ex Heavy Machinery\n   - **Timeframe:** 050626 to 270826\n\n6. **PT

### wbgt

In [3]:
import re
from datetime import datetime, timedelta

import requests
from bs4 import BeautifulSoup


def hhmm_to_datetime(hhmm: str) -> datetime:
    """Convert an HHMM string into today's datetime."""
    return datetime.now().replace(
        hour=int(hhmm[:2]),
        minute=int(hhmm[2:]),
        second=0,
        microsecond=0,
    )


response = requests.post(
    "https://t.me/s/armycat1_v2",
    headers={
        "X-Requested-With": "XMLHttpRequest",
    },
)
response.raise_for_status()
print(response)

# Telegram normally returns HTML here.
soup = BeautifulSoup(response.text, "html.parser")

message_elements = soup.select(
    ".tgme_widget_message_wrap .tgme_widget_message_text"
)

last_message = (
    message_elements[-1].get_text(strip=True)
    if message_elements
    else ""
)

cat1_times = re.findall(
    r"\((\d{4})-(\d{4})\)([\w,]+)",
    last_message,
)

cat1_sectors = []

for start, end, raw_sectors in cat1_times:
    sectors = raw_sectors.split(",")

    start_datetime = hhmm_to_datetime(start)
    end_datetime = hhmm_to_datetime(end)

    # Handle time ranges that cross midnight.
    if end_datetime < start_datetime:
        end_datetime += timedelta(days=1)

    for sector in sectors:
        cat1_sectors.append(
            {
                "sector": re.sub(r"^0", "", sector),
                "start": start_datetime,
                "end": end_datetime,
                "src": "telegram",
            }
        )

print(cat1_sectors)

<Response [200]>
[]


In [44]:
import requests
from bs4 import BeautifulSoup

url = "https://t.me/s/ArmyCAT1_v2"

html = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"}
).text

soup = BeautifulSoup(html, "html.parser")

# Get all messages
messages = soup.select(".tgme_widget_message_text")

# Latest message
latest_message = messages[-1].get_text("\n", strip=True)

print(latest_message)

[CAT Status Update]
⚡️
All Sectors Clear (1730-2100)


In [30]:
import requests
from bs4 import BeautifulSoup

url = "https://t.me/s/armynaws"

html = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"}
).text

soup = BeautifulSoup(html, "html.parser")

# Get all messages
messages = soup.select(".tgme_widget_message_text")

# Latest message
latest_message = messages[-1].get_text("\n", strip=True)

print(latest_message)

WBGT Update - 03 Aug 2026 1815hrs
⚠
CUTOFF --- Work : Rest (No Strenuous Training)
NIL
⚫
BLACK --- Work : Rest (15:30)
NIL
🔴
RED --- Work : Rest (30:30)
NIL
🟡
YELLOW --- Work : Rest (30:15)
NIL
🟢
GREEN --- Work : Rest (45:15)
• Bedok Camp, 30.5℃
• Chong Pang Camp, 30.3℃
• D2/D3/MUTF, 30.6℃
• MA 8 - Upper Mandai, 30.0℃
⚪
WHITE --- Work : Rest (60:15)
• A1/A2, 28.3℃
• Bukit Panjang/Hillview Camp/Gombak Base, 29.7℃
• C, 27.8℃
• Changi Airbase (East), 29.7℃
• Changi Airbase (West), 29.5℃
• Changi Naval Base, 28.9℃
• Choa Chu Kang Camp, 29.0℃
• Clementi Camp, 28.9℃
• D1/E/TP F (N)/TP F (S)/TP 8 - Area E, 29.6℃
• Dieppe Barracks/Khatib Camp, 29.4℃
• Hendon Camp, 29.0℃
• Jurong Camp I and II, 29.0℃
• Keat Hong Camp, 29.0℃
• Lim Chu Kang Camp I/II, 28.6℃
• MA 2, 29.4℃
• MA 8 - Lower Mandai, 28.5℃
• Maju Camp, 28.4℃
• Mandai Hill Camp, 29.8℃
• Mandai West Camp, 29.2℃
• Mowbray/Kranji Camp II and III, 29.6℃
• Nee Soon Camp, 28.6℃
• Pasir Laba Camp, 28.4℃
• Pasir Ris Camp, 29.8℃
• Rifle Range Roa

In [33]:
import re
from datetime import datetime

def extract_wbgt(text: str, camp: str) -> dict | None:
    # Extract update timestamp
    header_match = re.search(
        r"WBGT Update\s*-\s*(\d{2}\s+[A-Za-z]{3}\s+\d{4}\s+\d{4})hrs",
        text,
    )

    update_time = header_match.group(1) if header_match else None

    # Extract the camp's WBGT
    camp_pattern = re.compile(
        rf"•\s*{re.escape(camp)},\s*(\d+(?:\.\d+)?)℃",
        flags=re.IGNORECASE,
    )

    camp_match = camp_pattern.search(text)

    if not camp_match:
        return None

    # Find the most recent colour heading before the camp
    preceding_text = text[:camp_match.start()]

    headings = re.findall(
        r"^(CUTOFF|BLACK|RED|YELLOW|GREEN|WHITE)\s+---",
        preceding_text,
        flags=re.MULTILINE,
    )

    return {
        "camp": camp,
        "wbgt": float(camp_match.group(1)),
        "category": headings[-1] if headings else None,
        "updated_at": update_time,
    }

result = extract_wbgt(latest_message, "Sungei Gedong Camp")
print(result)

{'camp': 'Sungei Gedong Camp', 'wbgt': 29.2, 'category': 'WHITE', 'updated_at': '03 Aug 2026 1815'}


In [9]:
def get_latest_message(channel):
    url = channel

    html = requests.get(
        url,
        headers={"User-Agent": "Mozilla/5.0"}
    ).text

    soup = BeautifulSoup(html, "html.parser")

    # Get all messages
    messages = soup.select(".tgme_widget_message_text")

    # Latest message
    latest_message = messages[-1].get_text("\n", strip=True)
    return latest_message

In [ ]:
CHANNELS = {"wbgt": "https://t.me/s/armynaws", "cat": "https://t.me/s/ArmyCAT1_v2"}

In [17]:
import pprint
import json

In [ ]:
resp = requests.get("https://api-open.data.gov.sg/v2/real-time/api/psi")
resp_clean = json.loads(resp.text)
resp_clean['data']
resp_clean['data']['items'][0]['readings']['psi_twenty_four_hourly']['north']

55

In [23]:
resp_clean['data']['items'][0]['timestamp']

'2026-08-03T18:00:00+08:00'

In [28]:
def get_psi_north():
    resp = requests.get("https://api-open.data.gov.sg/v2/real-time/api/psi")
    resp_clean = json.loads(resp.text)
    psi = resp_clean['data']['items'][0]['readings']['psi_twenty_four_hourly']['north']
    time = resp_clean['data']['items'][0]['timestamp']
    return "North", time, psi

In [29]:
get_psi_north()

('North', '2026-08-03T18:00:00+08:00', 55)

In [42]:
import re


def get_cat_status(message: str, target_sector: str = "3N") -> dict:
    target_sector = target_sector.upper().strip()

    # Special case: All sectors clear
    m = re.search(
        r"All\s+Sectors\s+Clear\s*\((\d{4})-(\d{4})\)",
        message,
        re.IGNORECASE,
    )
    if m:
        return {
            "sector": target_sector,
            "status": "CAT 3",
            "start": m.group(1),
            "end": m.group(2),
        }

    # CAT 1 / CAT 2 sections
    section_pattern = re.compile(
        r"CAT\s*([12])\s*:\s*(.*?)(?=CAT\s*[12]\s*:|\Z)",
        re.IGNORECASE | re.DOTALL,
    )

    block_pattern = re.compile(
        r"\((\d{4})-(\d{4})\)\s*\n+([A-Za-z0-9,\s]+)"
    )

    for cat, section in section_pattern.findall(message):
        for start, end, raw_sectors in block_pattern.findall(section):
            sectors = {
                s.strip().upper()
                for s in raw_sectors.split(",")
                if s.strip()
            }

            # Exact match (3N != 13N)
            if target_sector in sectors:
                return {
                    "sector": target_sector,
                    "status": f"CAT {cat}",
                    "start": start,
                    "end": end,
                }

    # Sector wasn't mentioned, so it's CAT 3.
    return {
        "sector": target_sector,
        "status": "CAT 3",
        "start": None,
        "end": None,
    }

In [47]:
get_cat_status(latest_message)

{'sector': '3N', 'status': 'CAT 3', 'start': '1730', 'end': '2100'}

''